# Step 3 — Fine-Tuning SQLCoder 7B

**Important:** Ollama does not support fine-tuning directly.
We fine-tune using HuggingFace + LoRA, save the adapter, then re-register it with Ollama.

Flow:
1. Load sqlcoder-7b-2 weights from HuggingFace (same weights Ollama uses)
2. Apply QLoRA and train on your 80k rows
3. Save adapter → merge → export to Ollama as `sqlcoder-finetuned`

**M4 16GB settings:** batch=1, grad_accum=16, fp16, max_seq=512

In [3]:
import sys
!{sys.executable} -m pip install -q transformers accelerate peft datasets trl
print('✅ Done')

✅ Done


In [4]:
import os, json, csv, time, re
from pathlib import Path
import torch

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

DEVICE      = 'mps' if torch.backends.mps.is_available() else 'cpu'
MODEL_NAME  = 'defog/sqlcoder-7b-2'
MAX_SEQ_LEN = 512
OUTPUT_DIR  = Path('checkpoints')
OUTPUT_DIR.mkdir(exist_ok=True)
Path('logs').mkdir(exist_ok=True)

print(f'Device  : {DEVICE.upper()}')
print(f'PyTorch : {torch.__version__}')

Device  : MPS
PyTorch : 2.11.0


In [5]:
from transformers import AutoTokenizer

print(f'Loading tokenizer: {MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
print(f'✅ Tokenizer ready — vocab: {tokenizer.vocab_size:,}')

Loading tokenizer: defog/sqlcoder-7b-2 ...


✅ Tokenizer ready — vocab: 32,016


In [7]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

# 1. Define the 4-bit configuration to save memory
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

print(f'Loading model → {DEVICE.upper()} (4-bit quantized) ...')

# 2. Load the model with memory-saving flags
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",          # Let the library handle the memory mapping
    low_cpu_mem_usage=True      # Reduces RAM spikes during loading
)

Loading model → MPS (4-bit quantized) ...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/Users/zenalyst/Documents/llm-benchmark/venv/lib/python3.14/site-packages/bitsandbytes/backends/default/ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [8]:
from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj'],
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print('✅ LoRA applied')

trainable params: 4,194,304 || all params: 6,742,740,992 || trainable%: 0.0622
✅ LoRA applied


In [16]:
import csv
import time
import os
import torch
from transformers import (
    TrainingArguments, Trainer,
    DataCollatorForLanguageModeling, TrainerCallback
)
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model, PeftModel

# 1. Safety check for LoRA
if not isinstance(model, PeftModel):
    print("Applying LoRA adapters...")
    model.gradient_checkpointing_enable()
    model = prepare_model_for_kbit_training(model)
    peft_config = LoraConfig(
        r=8, 
        lora_alpha=32, 
        target_modules=["q_proj", "v_proj"], 
        lora_dropout=0.05, 
        bias="none", 
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, peft_config)
else:
    print("Model already has LoRA adapters. Skipping re-wrap.")

# 2. Setup Logging
log_path = 'logs/train_log.csv'
os.makedirs('logs', exist_ok=True)
log_file = open(log_path, 'w', newline='')
log_writer = csv.writer(log_file)
log_writer.writerow(['step', 'train_loss', 'eval_loss', 'lr', 'epoch'])

class CsvLogger(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            log_writer.writerow([
                state.global_step,
                logs.get('loss'),
                logs.get('eval_loss'),
                logs.get('learning_rate'),
                logs.get('epoch'),
            ])
            log_file.flush()

# 3. Updated Training Arguments (Cleaned for M4 compatibility)
training_args = TrainingArguments(
    output_dir='checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=500,
    save_strategy='steps',
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    fp16=False,
    bf16=False, 
    dataloader_pin_memory=False,
    report_to='none',
    # Removed use_mps_device as it is now automatic
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    callbacks=[CsvLogger()],
)

print('🚀 Training starting on M4 (MPS)...')
start = time.time()
try:
    trainer.train()
finally:
    # This ensures the file closes even if you stop the training manually
    elapsed = int(time.time() - start)
    log_file.close()
    h, m = divmod(elapsed // 60, 60)
    print(f'\n✅ Session ended. Total time: {h}h {m}m')

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Model already has LoRA adapters. Skipping re-wrap.


TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'

In [12]:
import csv
import time
from transformers import (
    TrainingArguments, Trainer,
    DataCollatorForLanguageModeling, TrainerCallback
)
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

# 1. Prepare model for low-memory fine-tuning (Required for 16GB Macs)
# This keeps the base model in 4-bit and adds trainable adapters
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# LoRA configuration (Low-Rank Adaptation)
peft_config = LoraConfig(
    r=8, 
    lora_alpha=32, 
    target_modules=["q_proj", "v_proj"], # Common for Llama/Mistral
    lora_dropout=0.05, 
    bias="none", 
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

# CSV loss logger setup
log_path = 'logs/train_log.csv'
import os
os.makedirs('logs', exist_ok=True)
log_file = open(log_path, 'w', newline='')
log_writer = csv.writer(log_file)
log_writer.writerow(['step', 'train_loss', 'eval_loss', 'lr', 'epoch'])

class CsvLogger(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            log_writer.writerow([
                state.global_step,
                logs.get('loss'),
                logs.get('eval_loss'),
                logs.get('learning_rate'),
                logs.get('epoch'),
            ])
            log_file.flush()

# 2. Updated Training Arguments for Apple Silicon
training_args = TrainingArguments(
    output_dir='checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16, # High accumulation helps stability on small batches
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    weight_decay=0.01,
    logging_steps=10,
    evaluation_strategy='steps',
    eval_steps=500,
    save_strategy='steps',
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    # Apple Silicon specific tweaks:
    fp16=False,
    bf16=False, # M4 supports bf16, but stick to False if using 4-bit/MPS for stability
    dataloader_pin_memory=False,
    report_to='none',
    # use_mps_device is usually handled by device_map="auto", 
    # but we'll leave it if your environment requires it.
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    callbacks=[CsvLogger()],
)

print(f'🚀 Training started — M4 16GB using LoRA + 4-bit')
print(f'Logs: {log_path} | Checkpoints: checkpoints/')

start = time.time()
trainer.train()
elapsed = int(time.time() - start)
log_file.close()

h, m = divmod(elapsed // 60, 60)
print(f'\n✅ Training done in {h}h {m}m')

/Users/zenalyst/Documents/llm-benchmark/venv/lib/python3.14/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/Users/zenalyst/Documents/llm-benchmark/venv/lib/python3.14/site-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [ ]:
# Save LoRA adapter
model.save_pretrained('checkpoints/best_adapter')
tokenizer.save_pretrained('checkpoints/best_adapter')
print('✅ Adapter saved → checkpoints/best_adapter')

# Show final training loss
import pandas as pd
log_df = pd.read_csv('logs/train_log.csv').dropna(subset=['train_loss'])
print(f'\nFinal train loss : {log_df["train_loss"].iloc[-1]:.4f}')
print(f'Best eval loss   : {log_df["eval_loss"].dropna().min():.4f}')
print()
print('✅ Step 3 done — paste output in chat')

In [ ]:
# Merge LoRA weights into base model and export for Ollama
from peft import PeftModel
from transformers import AutoModelForCausalLM

print('Merging LoRA adapter into base model...')
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map={'': 'cpu'},   # merge on CPU to avoid MPS OOM
    trust_remote_code=True,
)
merged = PeftModel.from_pretrained(base, 'checkpoints/best_adapter')
merged = merged.merge_and_unload()
merged.save_pretrained('checkpoints/merged_model', safe_serialization=True)
tokenizer.save_pretrained('checkpoints/merged_model')
print('✅ Merged model saved → checkpoints/merged_model')

# Write Modelfile for Ollama
modelfile = """FROM ./checkpoints/merged_model
PARAMETER temperature 0
PARAMETER num_predict 200
SYSTEM \"You are SQLCoder, an expert at generating SQL queries from natural language.\"
"""
with open('Modelfile', 'w') as f:
    f.write(modelfile)

print('\n✅ Modelfile written')
print('\nTo register with Ollama, run in terminal:')
print('  ollama create sqlcoder-finetuned -f Modelfile')
print('  ollama run sqlcoder-finetuned')

SyntaxError: invalid syntax (3784765619.py, line 1)